# 1 · The corpus

What is in `data/`, where it came from, and what the formalised seed looks like.

Four bodies of text ship with this repository:

| tradition | provisions | language | licence |
|---|---:|---|---|
| Roman & canon — Digest, Codex, Theodosian Code, Gaius, Institutes, XII Tables, Decretals | ~30,300 | Latin | Public Domain Mark 1.0 |
| Chinese — Tang / Ming / Qing codes, Song and Qing case reports, Legalists | ~16,800 | Classical Chinese | unspecified (works PD by age) |
| Rabbinic & biblical — Mishnah, Talmud, Exodus | ~13,400 | English | CC0 / CC-BY / CC-BY-NC / PD |
| English — Magna Carta | 63 | Latin | Public Domain Mark 1.0 |

All were fetched from GitHub mirrors (`cltk/lat_text_latin_library`,
`garychowcmu/daizhigev20`, `Sefaria/Sefaria-Export-Archive`).  A fifth group —
Hammurabi, Eshnunna, the Hittite laws, Gortyn, the Anglo-Saxon codes, Bracton,
the scienter cases and the Animals Act 1971 — is carried as citations and
editorial restatements only, because the sandbox this was built in could not
reach those sources. `python -m nomos.fetch.restricted --list` shows them and
will fetch them anywhere with ordinary network access.

**A word on the non-commercial texts.** The William Davidson Talmud is
CC-BY-NC. If you need a corpus you can use commercially, rebuild with
`python -m nomos.build_corpus --no-nc`; you lose the Talmud and keep everything
else.

In [1]:
import os, sys, json, subprocess
from pathlib import Path

# Locate the repository root whether this runs from notebooks/ or the root.
ROOT = Path.cwd()
while not (ROOT / "src" / "nomos").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)
print("repo root:", ROOT)

# The built corpus is a reproducible artefact and is not shipped in the
# archive (it is 13 MB gzipped and rebuilds from data/raw in about a minute).
if not any((ROOT / "data/corpus").glob("provisions.jsonl*")):
    print("building the corpus from data/raw ...")
    subprocess.run([sys.executable, "-m", "nomos.build_corpus"], check=True,
                   env={**os.environ, "PYTHONPATH": str(ROOT / "src")})


repo root: /home/claude/nomos


In [2]:
from nomos.schema import read_jsonl
from collections import Counter

provisions = list(read_jsonl("data/corpus/provisions.jsonl"))
print(f"{len(provisions):,} provisions, {sum(len(p['text'] or '') for p in provisions)/1e6:.1f}M characters")

by_work = Counter(p["work"] for p in provisions)
for work, n in by_work.most_common():
    lic = next(p["source"]["license"] for p in provisions if p["work"] == work)
    print(f"  {n:6,}  {work:28s} [{lic}]")

60,595 provisions, 21.4M characters
  20,770  Digesta Iustiniani           [Public Domain Mark 1.0]
   9,097  通典 (Comprehensive Institutions) [unspecified (underlying works public domain by age)]
   5,270  Codex Iustinianus            [Public Domain Mark 1.0]
   4,301  Bava Kamma                   [CC-BY-NC]
   3,793  Bava Batra                   [CC-BY-NC]
   3,647  Bava Metzia                  [CC-BY-NC]
   2,916  Codex Theodosianus           [Public Domain Mark 1.0]
   2,331  大清律例 (Great Qing Code with Substatutes) [unspecified (underlying works public domain by age)]
   1,210  Exodus                       [Public Domain]
   1,076  刑案匯覽三編 (Conspectus of Penal Cases, 3rd series) [unspecified (underlying works public domain by age)]
     829  大明律集解附例 (Ming Code with Commentary and Substatutes) [unspecified (underlying works public domain by age)]
     752  Gai Institutiones            [Public Domain Mark 1.0]
     698  明代律例彙編 (Ming Statutes and Substatutes, compiled) [unspecified (und

### The Digest

Book 9 title 1 — *si quadrupes pauperiem fecisse dicatur*, "if a four-footed
beast is said to have done damage" — is the Roman goring-ox title. It opens by
deriving the action from the Twelve Tables and stating the noxal election, and
then, four fragments later, gives the condition that Exodus and the Mishnah
also single out: *bos cornu petere solitus*, an ox accustomed to gore.

In [3]:
idx = {p["canonical"]: p for p in provisions}
for c in ["D.9.1.1pr", "D.9.1.1.4", "D.9.2.44pr", "D.50.17.203"]:
    p = idx[c]
    print(f"[{p['citation']}]  {p['attribution']}")
    print("   ", p["text"][:400])
    print()

[Dig. 9.1.1pr]  Ulpianus 18 ad ed
    Si quadrupes pauperiem fecisse dicetur, actio ex lege duodecim tabularum descendit: quae lex voluit aut dari id quod nocuit, id est id animal quod noxiam commisit, aut aestimationem noxiae offerre.

[Dig. 9.1.1.4]  Ulpianus 18 ad ed
    Itaque, ut servius scribit, tunc haec actio locum habet, cum commota feritate nocuit quadrupes, puta si equus calcitrosus calce percusserit, aut bos cornu petere solitus petierit, aut mulae propter nimiam ferociam: quod si propter loci iniquitatem aut propter culpam mulionis, aut si plus iusto onerata quadrupes in aliquem onus everterit, haec actio cessabit damnique iniuriae agetur.

[Dig. 9.2.44pr]  Ulpianus 42 ad sab
    In lege aquilia et levissima culpa venit.

[Dig. 50.17.203]  Pomponius 8 ad q. muc
    Quod quis ex culpa sua damnum sentit, non intellegitur damnum sentire.



### The Tang Code

653 CE, 502 articles, and the control case for the whole comparative argument:
a legal order with no contact with Rome, Babylon or Israel.

Article 207 makes the keeper of a dangerous animal mark and tether it, and the
official commentary quotes the Miscellaneous Ordinances on what that means —
*cut off both horns, hobble the feet, cut off both ears*. Hammurabi §251 holds
the owner liable where "he did not blunt its horns or tie up his ox."

Article 206 gives **half** the depreciation when beasts kill each other —
Exodus 21:35, Eshnunna §53 and the Mishnah's *ḥatzi nezek*, reached
independently.

And Qing Code art. 44 is a statutory rule governing reasoning by analogy.

In [4]:
for c in ["TL.207", "TL.206", "TL.204"]:
    r = idx[c]
    print(f"[{r['citation']}]")
    print("  律:", r["text"])
    if r["meta"].get("commentary"):
        print("  疏:", r["meta"]["commentary"][:220])
    print()

q = idx["DQ.44.00"]
print(f"[{q['citation']}]  斷罪無正條 — deciding a case with no exact provision")
print("  ", q["text"][:300])

[唐律疏議 art. 207 畜产蹋人]
  律: 诸畜产及噬犬有蹋人，而标帜羁绊不如法，若狂犬不杀者，笞四十；以故杀伤人者，以过失论。若故放令杀伤人者，减斗杀伤一等。
  疏: 【疏】议曰：依《杂令》：“畜产人者，截两角；蹋人者，绊足；人者，截两耳。”此为标帜羁绊之法。若不如法，并狂犬本主不杀之者，各笞四十。以不施标帜羁绊及狂犬不杀之故，致杀伤人者，以过失论。过失者，各依其罪从赎法。律无异文，总依凡法，不限尊贵，其赎一也。若本应轻者，听从本。其“故放令杀伤人者”，谓知犬及杂畜性能蹋及噬，而故放者，减斗杀伤一等。其犯贵贱、尊卑、长幼、亲属等，各依本犯应加减为罪。其畜产杀伤人，仍作他物伤人，保辜二十日，辜内

[唐律疏議 art. 206 犬杀伤畜产]
  律: 诸犬自杀伤他人畜产者，犬主偿其减价；馀畜自相杀伤者，偿减价之半。即故放令杀伤他人畜产者，各以故杀伤论。
  疏: 【疏】议曰：犬性噬，或自杀伤他人畜产。“犬主偿其减价”，以犬能噬，主须制之，为主不制，故令偿减价。“馀畜”，除犬之外，皆是。“自相杀伤者”，谓牛相杀，马相蹋死之类。假有甲家牛，杀乙家马，马本直绢十疋，为杀，估皮肉直绢两疋，即是减八疋绢，甲偿乙绢四疋，是名“偿减价之半”。“即故放令杀伤他人畜产者”，或犬性好噬猪羊，其牛马能相蹋，而故放者，责其故放，各与故杀伤罪同，谓同上条“故杀官私马牛者，徒一年半。计赃应重，若伤及杀馀畜产者，

[唐律疏議 art. 204 官私畜毁食官私物]
  律: 诸官私畜产，毁食官私之物，登时杀伤者，各减故杀伤三等，偿所减价；畜主备所毁。临时专制亦为主。馀条准此。
  疏: 【疏】议曰：畜产不限官私。或毁食官私之物者，毁谓有所唐突，或蹋之类。因其毁食，物主登时即杀伤者，各减前条“故杀伤”罪三等，若杀马牛，杖九十；其伤马牛及杀伤馀畜产，各计所减价，计赃准盗论减三等。如所杀马牛准所减价，当绢十五疋者，徒二年上减三等，合杖一百，如此计赃得罪重，即从重论。仍各偿所减价，畜主备所毁。假有一牛，直上绢五疋，毁食人物，平直上绢两疋，其物主登时伤杀此牛，出卖直绢三疋，计减二疋，牛主偿所损食绢二疋，物主酬所减牛价绢亦二疋之

[大清律例 44.00 断罪无正条]  斷罪無正條 — deciding a case with no exact provision
 

> 凡律令該載不盡事理，若斷罪無正條者，引律**比附**，應加應減，定擬罪名，申該上司
> 議定奏聞。若輒斷決，致罪有出入者，以故失論。

"Where the statutes do not exhaustively cover the matter, and there is no exact
provision, cite a statute **by analogy**, determine the appropriate increase or
decrease, settle the designation of the offence, and report it upward for
memorial to the throne. If an official decides it outright and the penalty
comes out too heavy or too light, he is punished as for misjudgment."

That is the gap `Reasoning/Analogy.lean` formalises as `no_strengthening` —
analogy fixes a bound, not a value — recognised and legislated about. 應加應減,
"the appropriate increase or decrease", *is* the gap. The Sages answered it
with *dayyo*; the Qing answered it with mandatory review and personal liability
on the judge.

### Mishnah Bava Kamma 1:1

This is the passage that made the project seem tractable. Read it as an
argument rather than a list: four paradigm cases, then each distinguished from
the others, then the common feature extracted in order to license extension to
cases not named. That is factor-based precedential constraint, described from
the inside, around 200 CE.

In [5]:
print(idx["Mishnah Bava Kamma 1:1"]["text"])
print()
print("--- and the dayyo dispute, m.BK 2:5 ---")
print(idx["Mishnah Bava Kamma 2:5"]["text"][:1200])

There are four primary causes of injury: the ox and the pit and the crop-destroying beast and fire. [The distinctive feature of] the ox is not like [that of] the crop-destroying beast, nor is [the distinctive feature of] either of these, which are alive, like [that of] fire, which is not alive; nor is [the distinctive feature of] any of these, whose way it is to go forth and do injury, like [that of] the pit, whose way it is not to go forth and do injury. What they have in common is that it is their way to do injury and that you are responsible for caring over them; and if one of them did injury whoever [is responsible] for the injury must make restitution [to the damaged party] with the best of his land.

--- and the dayyo dispute, m.BK 2:5 ---
“An ox which causes damage in the private domain of him that is injured” how is this so? If it gored, pushed, bit, lay down, or kicked in the public domain its owner pays only half damages. But if in the private domain of him that is injured, R

## The seed formalisations

43 holdings across four traditions, hand-made and machine-checked. The Lean
library is the source of truth; `python -m nomos.build_seed` exports it.

In [6]:
seed_rows = list(read_jsonl("data/seed/formalizations.jsonl"))
print(f"{len(seed_rows)} seed holdings")
print("by tradition:  ", dict(Counter(r["tradition"] for r in seed_rows)))
print("by fact pattern:", dict(Counter(r["fact_pattern"] for r in seed_rows)))
print("all well-formed:", all(r["well_formed"] for r in seed_rows))
print()
r = next(r for r in seed_rows if r["cite"] == "Ex 21:29")
print("text: ", r["text"])
print("situation:", r["situation"])
print("winner:   ", r["winner"])
print("remedy:   ", r["remedy"])
print("ratio:    ", r["reason"])
print()
print(r["lean"])

60 seed holdings
by tradition:   {'covenant': 12, 'rabbinic': 11, 'roman': 13, 'mesopotamian': 7, 'chinese': 8, 'english': 9}
by fact pattern: {'FP-GORE': 27, 'FP-GRAZE': 6, 'FP-PIT': 2, 'FP-FIRE': 2, 'FP-BAILMENT': 7, 'FP-CONSENT': 1, 'FP-WRONGFUL-DAMAGE': 11, 'FP-ASSAULT': 3, 'FP-BUILD': 1}
all well-formed: True

text:  But if the ox was wont to gore in time past, and warning hath been given to its owner, and he hath not kept it in, but it hath killed a man or a woman; the ox shall be stoned, and its owner also shall be put to death.
situation: ['harmOccurred', 'respondentsInstrument', 'harmToPerson', 'knownVice', 'warned', 'noPrecaution']
winner:    claimant
remedy:    Remedy.capital
ratio:     ['knownVice', 'warned', 'noPrecaution']

def holding : Precedent :=
  { cite := "Ex 21:29"
  , situation := ⟨[Factor.harmOccurred, Factor.respondentsInstrument, Factor.harmToPerson, Factor.knownVice, Factor.warned, Factor.noPrecaution]⟩
  , winner := Side.claimant
  , remedy := Remedy.capital

## Fact patterns

The alignment axis. A fact pattern is a question many orders had to answer; the
scenario is stable across cultures and the answer is not, and the gap between
those two is what we are trying to measure.

In [7]:
patterns = json.load(open("data/corpus/fact_patterns.json"))
for p in patterns:
    wits = ", ".join(f"{w['tradition']}: {w['cite']}" for w in p["witnesses"])
    print(f"{p['id']:22s} {p['name']}")
    print(f"{'':22s} {p['issue']}")
    print(f"{'':22s} {wits}")
    print()

FP-GRAZE               Grazing trespass
                       Does A answer to B, and in what?
                       mesopotamian: LH §57-58, covenant: Ex 22:4, rabbinic: m.BK 1:1 (shen), 2:2, 6:1, roman: D.19.5.14.3; actio de pastu pecoris, chinese: 唐律疏議 arts. 204, 209

FP-GORE                Goring animal
                       Does A answer, and does prior notice of the vice change the answer?
                       mesopotamian: LE §53-55; LH §250-252, covenant: Ex 21:28-36, rabbinic: m.BK 1:4, 2:4, 4:9, roman: XII Tab.; D.9.1 (actio de pauperie), chinese: 唐律疏議 arts. 206, 207, english: Alfred Af El. ~21; May v Burdett (1846); Animals Act 1971 s.2(2)

FP-PIT                 Hazard left in place
                       Does A answer for a static hazard he created but did not direct?
                       covenant: Ex 21:33-34, rabbinic: m.BK 1:1 (bor), 3:1, 5:5, roman: D.9.2.28; D.9.3 (de effusis)

FP-FIRE                Spreading fire
                       Does A answer for the s

## Coherence diagnostics

The same constraint relation the Lean library proves theorems about, computed
in Python over the seed corpus. Two things are worth looking at: whether each
tradition contradicts itself on the facts it addressed, and what happens when
you try to merge two traditions into one body of precedent.

The Covenant Code result below is the one we did not expect.

In [8]:
from nomos.analogy import Holding, conflicts_at, incoherence_report, is_open

def load(rows, trad=None):
    return [Holding(cite=r["cite"], situation=r["situation"], winner=r["winner"],
                    remedy=r["remedy"], reason=r["reason"], tradition=r["tradition"])
            for r in rows if trad is None or r["tradition"] == trad]

seed = load(seed_rows)
for trad in ["covenant", "rabbinic", "roman", "mesopotamian", "chinese", "english"]:
    c = load(seed_rows, trad)
    rep = incoherence_report(c)
    print(f"{trad:14s} {len(c):3d} holdings   self-conflicts: {len(rep)}")
    for r in rep[:2]:
        for cf in r["conflicts"]:
            print(f"                 {cf['claimant']}  (claimant)  vs  {cf['respondent']}  (respondent)")

covenant        12 holdings   self-conflicts: 2
                 Ex 21:35  (claimant)  vs  Ex 21:28  (respondent)
                 Ex 21:35  (claimant)  vs  Ex 21:28  (respondent)
rabbinic        11 holdings   self-conflicts: 0
roman           13 holdings   self-conflicts: 0
mesopotamian     7 holdings   self-conflicts: 2
                 LE §53 [editorial restatement; text not shipped]  (claimant)  vs  LH §250 [editorial restatement; text not shipped]  (respondent)
                 LE §53 [editorial restatement; text not shipped]  (claimant)  vs  LH §250 [editorial restatement; text not shipped]  (respondent)
chinese          8 holdings   self-conflicts: 0
english          9 holdings   self-conflicts: 0


The Covenant Code conflict is between **Exodus 21:35** (an ox kills another
man's ox: the loss is divided, so the claimant recovers something) and
**Exodus 21:28** (an ox kills a *man*: "the owner of the ox shall be quit", so
the claimant recovers nothing).

The second situation has every claimant-side factor the first has, plus
`harmToPerson`, and yields less. On the reason model that is a contradiction.

We think it is a finding rather than a bug, and what it localises is the
interesting part. Ex 21:28 is not operating on a compensation logic at all: the
ox is stoned and its flesh may not be eaten, which is how a thing that has shed
human blood is treated (compare Gen 9:5), and the owner being *naqi* is
acquittal of bloodguilt rather than denial of a debt. A factor vocabulary built
for loss-allocation cannot see a sacral category, so it reports the seam as a
contradiction — localised to two verses, reproducibly, in a way a prose
comparison would glide over.

Now the cross-tradition merge.

In [9]:
cov, rom, rab = load(seed_rows, "covenant"), load(seed_rows, "roman"), load(seed_rows, "rabbinic")

by_trad = {}
for h in seed:
    by_trad.setdefault(h.tradition, []).append(h)
names = sorted(by_trad)
respondents = set()
print(f"{'pair':30s} conflicts")
for i, a in enumerate(names):
    for b in names[i + 1:]:
        merged = by_trad[a] + by_trad[b]
        cross = set()
        for h in merged:
            for p, c in conflicts_at(merged, h.situation):
                if (p in by_trad[a]) != (c in by_trad[a]):
                    cross.add((p.cite, c.cite)); respondents.add(c.cite)
        mark = "" if not cross else "   <- " + "; ".join(sorted({c for _, c in cross}))
        print(f"  {a[:13]:13s}+{b[:13]:13s} {len(cross):3d}{mark}")
print()
print("respondent-side holdings in EVERY cross-tradition conflict:")
for r in sorted(respondents):
    print("   ", r)

pair                           conflicts
  chinese      +covenant        1   <- Ex 21:28
  chinese      +english         0
  chinese      +mesopotamian    1   <- LH §250 [editorial restatement; text not shipped]
  chinese      +rabbinic        0
  chinese      +roman           0
  covenant     +english         1   <- Ex 21:28
  covenant     +mesopotamian    2   <- Ex 21:28; LH §250 [editorial restatement; text not shipped]
  covenant     +rabbinic        2   <- Ex 21:28
  covenant     +roman           1   <- Ex 21:28
  english      +mesopotamian    1   <- LH §250 [editorial restatement; text not shipped]
  english      +rabbinic        0
  english      +roman           0
  mesopotamian +rabbinic        2   <- LH §250 [editorial restatement; text not shipped]
  mesopotamian +roman           1   <- LH §250 [editorial restatement; text not shipped]
  rabbinic     +roman           0

respondent-side holdings in EVERY cross-tradition conflict:
    Ex 21:28
    LH §250 [editorial restatement

**This is the result we did not expect.**

Five of the fifteen pairs have no conflicts at all — China with Rome, China
with the Mishnah, China with England, England with Rome, England with the
Mishnah. And every conflict in the remaining ten has one of exactly two
holdings on the respondent's side:

* **Ex 21:28** — an ox gores a man, nothing known against it: *the owner shall
  be quit*.
* **LH §250** — an ox gores a man in the street: *this case has no penalty*.

Those are the same rule. Babylon and Israel exempt the keeper of an
un-forewarned animal; Rome, the Mishnah, Tang China and England all make him
answer something. The fault line in the entire corpus runs between *custody
alone grounds liability* and *custody plus notice does*, and it falls exactly
where two geographically adjacent traditions part company from four others.

A vocabulary loose enough to fit anything would have reported no fault line at
all. Proved as `incoherence_is_the_no_notice_no_liability_rule` in
`Nomos/Bench/GoringOx.lean`.

## Retrieval

How the pipeline chooses which precedents to show a model. Not by similarity —
by *constraint distance*. Binding precedents first, then near misses at
distance one, with the fact that blocks each one named.

In [10]:
from nomos.analogy import retrieve

query = ["harmOccurred", "respondentsInstrument", "harmToPerson",
         "knownVice", "warned", "properPrecaution"]
print("query situation:", query)
print("open under the seed corpus?", is_open(seed, query))
print()
for r in retrieve(seed, query, k=6):
    print(f"[{r.kind:12s}] {r.holding.cite}  ({r.holding.tradition})")
    print(f"               {r.why}")
    print()

query situation: ['harmOccurred', 'respondentsInstrument', 'harmToPerson', 'knownVice', 'warned', 'properPrecaution']
open under the seed corpus? True

[near-miss   ] Ex 21:32  (covenant)
               would bind but for 'properPrecaution', a counter-reason the precedent never faced

[near-miss   ] LH §252 [editorial restatement; text not shipped]  (mesopotamian)
               would bind but for 'properPrecaution', a counter-reason the precedent never faced

[near-miss   ] m.BK 1:4, 4:9 (mu'ad)  (rabbinic)
               would bind but for 'properPrecaution', a counter-reason the precedent never faced

[near-miss   ] May v Burdett (1846) 9 QB 101 [editorial restatement]  (english)
               would bind but for 'properPrecaution', a counter-reason the precedent never faced

[near-miss   ] 唐律疏議 art. 206 (犬殺傷畜產) -- dog  (chinese)
               would bind but for 'properPrecaution', a counter-reason the precedent never faced

[near-miss   ] D.9.1.1.4 (Ulpian, citing Servius)  (roman